# Methoden en Technieken -- Blok 2

## Datapunt 2a,

In deze opdracht worden de volgende leeruitkomsten getoetst, relevante termen zijn **dik** gedrukt:
- A2: Je stelt voor een AI-oplossing juridische, ethische, organisatorische, **functionele en
technische requirements** op.
- B1: Je **verkent en prepareert een dataset voor het trainen en testen van een AI-model en
kan de voor- en nadelen van het gebruik van een bestaande dataset onderbouwen**, rekening
houdend met technische en ethische randvoorwaarden.
- B2: Je **stelt op basis van requirements en data een geschikte architectuur voor een AI-
oplossing op en selecteert daarvoor passende AI-technieken gebruik makend van bijvoorbeeld**
machine learning, **deep learning**, kennisrepresentatie, computer vision en **natural language
processing**.
- B3: Je **ontwikkelt een nieuw** of voorgetraind **AI-model volgens een iteratief en systematisch
proces**.
- C2: **Je evalueert en beoordeelt de kwaliteit van een AI-model aan de hand van kwaliteitscriteria die in het vakgebied erkend worden** zoals robustness, **performance**, scalability,
explainability, **model complexity** en resource demand.

<hr>

- Dit is een individuele opdracht. Het is toegestaan met elkaar de opdracht te bespreken, maar iedereen schrijft zijn/haar eigen code en antwoorden. 
- Mocht je code van het internet hebben overgenomen/aangepast, vermeld dan de bron in het commentaar.
- Mocht je naar een bron verwijzen voor jouw argumentatie, vermeld dan de bron *en* copy-paste de specifieke uitspraak uit de bron waar je het argument op baseert. 
- Het gebruik van generatieve AI (Chat-GPT, etc...) is *alleen* toegestaan bij de voorbereiding op de opdracht. Denk hierbij aan activiteiten zoals brainstormen, of het maken van een plan. Het is niet toegestaan code of tekst van generative AI over te nemen. (Level 2 van de AI Assessment Scale)



## De opdracht

In de slides van les 3 staat hoe een Engels - Nederlands vertaalmodel gemaakt wordt met een transformer-encoder en -decoder. Zie ook hoofdstuk 15 uit Deep Learning with Python (derde editie).

### Deel 1
#### Deel 1a
Maak dit model na en fit dit model.

# Deel 1a — Het vertaalmodel uit les 3 nabouwen

## Voorwoord en aanpak

In deze opdracht bouw ik een Transformer-model dat Engelse zinnen naar het Nederlands vertaalt. In eerdere M&T-opdrachten merkte ik dat een werkend model op zichzelf niet genoeg is: ik moet ook kunnen uitleggen waarom ik voor een bepaalde aanpak kies, hoe ik met de data omga en wat de uitkomsten uiteindelijk betekenen. Daarom wil ik de opdracht niet alleen technisch uitvoeren, maar ook systematisch onderbouwen.

De opdracht verwijst expliciet naar het encoder-decodermodel uit **les 3**. Ik begin daarom met het reproduceren van die implementatie, zonder meteen de architectuur te verbeteren. Daarmee maak ik een **baseline**: een vast uitgangspunt waarmee ik in deel 2 kan vaststellen of een aanpassing daadwerkelijk iets oplevert. De centrale gedachte is dat ik niet achteraf een verbetering wil claimen zonder een eerlijke vergelijking.

Ik werk in vier stappen. Eerst onderzoek ik de opbouw van de zinsparen en maak ik een train-, validatie- en testsplit. Vervolgens zet ik tekst om naar numerieke representaties, bouw ik de encoder-decoder-Transformer en train ik het model. Als laatste controleer ik de learning curves en enkele gegenereerde vertalingen. Op basis van die resultaten kan ik pas bepalen welke verbetering relevant is.

**Herkomst van deze baseline:** `Blok2_Les03.ipynb`, onderdelen *Eng-Ned vertaaldataset*, *Transformer encoder*, *Transformer decoder*, *PositionalEmbedding* en *The transformer, training*. De les vermeldt dat de code grotendeels is gebaseerd op de notebooks bij F. Chollet, *Deep Learning with Python*, hoofdstuk 15 ([boek en lesmateriaal](https://deeplearningwithpython.io/); [notebooks](https://github.com/fchollet/deep-learning-with-python-notebooks)). De bronvermelding betreft de voorbeeldimplementatie, niet een onafhankelijke validatie van de prestaties.

## Onderzoekskader: wat moet de baseline aantonen?

De taak is *sequence-to-sequence learning*: een reeks Engelse tokens wordt omgezet naar een reeks Nederlandse tokens. Voor de baseline stel ik de volgende controleerbare eisen op. Dit zijn **operationele eisen voor de eerste implementatie**, nog geen beweringen over de behaalde prestaties.

| ID | Eis | Hoe ik deze controleer |
|---|---|---|
| F1 | Het model ontvangt een Engelse zin en voorspelt Nederlandse tokens. | De invoer-/uitvoervormen en voorbeeldvertalingen controleren. |
| F2 | Het model leert uitsluitend van de trainingsset. | De dataverdeling en de plaats van `adapt()` en `fit()` controleren. |
| T1 | De architectuur en instellingen zijn herleidbaar tot les 3. | Modeloverzicht en hyperparameters vastleggen. |
| T2 | Het trainen levert interpreteerbare leer- en validatiecurves op. | Loss en token-accuracy per epoch visualiseren. |
| T3 | Het experiment kan worden herhaald. | Seed, splitsing, instellingen en trainingsgeschiedenis vastleggen. |

Voor deze eerste reproductie spreek ik nog geen willekeurige minimale accuracy af. Wat **'goed genoeg'** betekent, bepaal ik later aan de hand van de gemeten validatieprestaties, de vertaalkwaliteit en de fouttypen. Daarmee voorkom ik dat een succescriterium alleen op een gevoel of op een achteraf gekozen score is gebaseerd.

## 1. Bibliotheken en reproduceerbaarheid

Ik gebruik TensorFlow en Keras, omdat het lesvoorbeeld op deze frameworks is gebaseerd. Voor de analyse van de resultaten gebruik ik NumPy, pandas en Matplotlib. Ik leg een vaste random seed vast, zodat de data-indeling en een deel van de stochasticiteit reproduceerbaar zijn. De seed garandeert niet dat iedere computer exact dezelfde trainingresultaten geeft, bijvoorbeeld door verschillen in hardware of numerieke uitvoering.

**Uitvoering:** de onderstaande code gebruikt het notebook als werkomgeving; lokale data, modelbestanden en figuren worden in de projectstructuur opgeslagen.

In [ ]:
import random
import string
import re
import json
import zipfile
from pathlib import Path

import keras
from keras import layers, ops
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 1407
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = Path("data/raw")
MODEL_DIR = Path("models")
RESULTS_DIR = Path("results")
FIGURES_DIR = RESULTS_DIR / "figures"
for folder in (DATA_DIR, MODEL_DIR, RESULTS_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Dataset inlezen en verdelen

De opdracht gebruikt de Engels-Nederlandse zinsparen van [ManyThings / Anki](https://www.manythings.org/anki/). In deze repository staat het bronbestand in `data/raw/nld-eng/nld.txt`. Per regel staan een Engelse zin, een Nederlandse vertaling en eventueel aanvullende broninformatie. Ik gebruik de eerste twee kolommen, omdat die direct aansluiten bij de vertaaltaak.

Ik voeg aan de Nederlandse doelzin `[start]` en `[end]` toe. Die tokens maken tijdens het genereren duidelijk waar een vertaling begint en wanneer deze klaar is. De Engelse invoer heeft deze speciale doelmarkeringen niet nodig.

### Waarom splits ik de data?

Ik gebruik dezelfde verhouding als in les 3: **70% training, 15% validatie en 15% test**. De trainingsset is voor het aanpassen van de modelgewichten, de validatieset voor het volgen van de prestaties en het vergelijken van latere iteraties. De testset blijft apart totdat het definitieve model is gekozen. Zo probeer ik te voorkomen dat ik mijn keuzes ongemerkt afstem op de eindbeoordeling.

Ik shuffle de zinsparen met een vaste seed. Daardoor blijft ieder Engels-Nederlands paar bij elkaar en is de splitsing na te maken. Een aandachtspunt is dat zinnen die bijna hetzelfde zijn mogelijk in verschillende subsets terechtkomen. Daarom is een willekeurige verdeling niet automatisch een garantie dat de sets inhoudelijk onafhankelijk zijn. Dat bespreek ik verder bij deel 1b.

De volgende code controleert eerst of de dataset op de juiste locatie staat en voert daarna de splitsing uit.

In [ ]:
# Pad komt overeen met de werkelijke mappenstructuur in de GitHub-repository.
# Controleer ook het huidige werkdirectory van Jupyter (meestal de projectroot).
dataset_path = DATA_DIR / "nld-eng" / "nld.txt"

if not dataset_path.is_file():
    raise FileNotFoundError(
        f"Dataset niet gevonden: {dataset_path.resolve()}\n"
        "Controleer of de repository is bijgewerkt met git pull origin main "
        "en of het notebook vanuit de projectmap MAAI_MenT draait."
    )

print("Dataset gevonden:", dataset_path.resolve())

text_pairs = []
with open(dataset_path, encoding="utf-8") as f:
    for line in f:
        fields = line.rstrip("\r\n").split("\t")
        if len(fields) >= 2 and fields[0] and fields[1]:
            text_pairs.append((fields[0], f"[start] {fields[1]} [end]"))

assert text_pairs, "Geen zinsparen ingelezen: controleer het datasetbestand."
random.Random(SEED).shuffle(text_pairs)
val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples
train_pairs = text_pairs[:train_samples]
val_pairs = text_pairs[train_samples:train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples:]
print("Totaal:", len(text_pairs), "| train:", len(train_pairs), "| validatie:", len(val_pairs), "| test:", len(test_pairs))
print("Voorbeeld:", train_pairs[0])

## 3. Eerste analyse van de dataset

Voordat ik een model ga trainen, wil ik begrijpen wat voor data ik gebruik. Bij een vertaalmodel zijn vooral het **aantal zinsparen**, de **zinslengtes** en de **dekking van de woordenschat** relevant. Een model met een maximale sequentielengte kan lange zinnen immers niet volledig verwerken wanneer die worden afgekapt.

Ik maak daarom twee visualisaties. De eerste laat zien hoeveel voorbeelden beschikbaar zijn voor training, validatie en test. De tweede toont hoe lang de Engelse en Nederlandse zinnen in de trainingsset zijn. Ik gebruik bewust alleen trainingszinnen voor de zinslengteanalyse, omdat ik de testset niet wil gebruiken om de modelinstellingen te kiezen.

Voor deze snelle visualisatie tel ik woorden via spaties. Dat is niet precies hetzelfde als de latere `TextVectorization`: leestekens en de speciale start- en eindtokens kunnen het aantal modeltokens beïnvloeden. De grafiek is dus een eerste verkenning, niet een exacte meting van hoeveel tokens na vectorisatie worden afgekapt.

**Na uitvoering controleren:** hoeveel zinnen zijn langer dan twintig woorden? En welke gevolgen zou de gekozen maximale lengte van twintig tokens kunnen hebben voor de vertaalkwaliteit? Ik neem dit mee als mogelijke beperking, maar verander de baseline-instelling nog niet.

In [ ]:
split_counts = {"Train": len(train_pairs), "Validatie": len(val_pairs), "Test (gereserveerd)": len(test_pairs)}
fig, ax = plt.subplots(figsize=(8, 3.5))
bars = ax.bar(split_counts.keys(), split_counts.values(), width=0.5)
ax.bar_label(bars, padding=3)
ax.set_title("Datasetverdeling — Engels–Nederlands")
ax.set_ylabel("Aantal zinsparen")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_datasetverdeling.png", dpi=160)
plt.show()

english_lengths = np.array([len(s.split()) for s, _ in train_pairs])
dutch_lengths = np.array([len(s.split()) for _, s in train_pairs])
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(english_lengths, bins=range(1, 51), alpha=0.6, label="Engels")
ax.hist(dutch_lengths, bins=range(1, 51), alpha=0.5, label="Nederlands incl. start/eind")
ax.axvline(20, linestyle="--", color="black", label="Max. invoerlengte = 20")
ax.set_xlim(0, 50)
ax.set_xlabel("Woorden per zin (ruwe spatiesplitsing)")
ax.set_ylabel("Frequentie")
ax.set_title("Verdeling van zinslengtes — trainingsset")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_zinslengtes.png", dpi=160)
plt.show()

## 4. Tekst voorbereiden voor de Transformer

Een Transformer kan niet rechtstreeks rekenen met woorden. Daarom zet ik de Engelse en Nederlandse tekst eerst om naar reeksen token-ID's. Ik gebruik hiervoor `TextVectorization`, net als in les 3.

De belangrijkste instellingen zijn een maximale woordenschat van **15.000 tokens** en een sequentielengte van **20** voor de Engelse invoer. Bij het Nederlands gebruik ik **21** posities, omdat ik tijdens training de invoer en de gewenste uitvoer één positie ten opzichte van elkaar verschuif. De speciale tokens `[start]` en `[end]` blijven behouden bij de standaardisatie.

### Waarom wordt de Nederlandse zin verschoven?

De decoder leert steeds een **volgend token** voorspellen op basis van de voorgaande tokens én de representatie van de Engelse zin. Stel dat de doelzin begint met `[start] ik ben`. Dan is de taak om bij de invoer `[start] ik` het volgende token `ben` te leren voorspellen. Dit principe staat bekend als *teacher forcing* tijdens training.

De `adapt()`-stap gebruik ik alleen op trainingszinnen. Anders zou de woordenschat mede worden bepaald door validatie- of testvoorbeelden. Dat zou weliswaar geen directe training van gewichten zijn, maar wel ongewenste informatieoverdracht vanuit data die ik juist apart wil houden.

Voor padding krijgen de doelposities met token-ID 0 een samplegewicht van nul, zodat deze bij de gewogen accuracy niet even zwaar meetellen als echte woorden. Ik moet later nog controleren hoe de loss en attention-maskers precies met padding omgaan.

**Bron voor gebruikte API en gedrag:** [Keras TextVectorization](https://keras.io/api/layers/preprocessing_layers/text/text_vectorization/) en de preprocessingcellen uit `Blok2_Les03.ipynb`.

In [ ]:
strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", ""
    )

vocab_size = 15000
sequence_length = 20

english_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
dutch_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)
train_english_texts = [pair[0] for pair in train_pairs]
train_dutch_texts = [pair[1] for pair in train_pairs]
english_tokenizer.adapt(train_english_texts)
dutch_tokenizer.adapt(train_dutch_texts)

In [ ]:
batch_size = 64

def format_dataset(eng, spa):
    eng = english_tokenizer(eng)
    spa = dutch_tokenizer(spa)
    features = {"english": eng, "dutch": spa[:, :-1]}
    labels = spa[:, 1:]
    sample_weights = labels != 0
    return features, labels, sample_weights

def make_dataset(pairs):
    eng_texts, spa_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    spa_texts = list(spa_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.cache().shuffle(2048, seed=SEED).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

for features, targets, weights in train_ds.take(1):
    print('Encoder:', features['english'].shape, 'Decoder:', features['dutch'].shape, 'Targets:', targets.shape)
    print('Paddinggewicht (eerste 10):', weights[0, :10].numpy())

## 5. Waarom een Transformer encoder-decoder?

De opdracht vraagt om het model uit les 3 na te bouwen. Ik wil echter ook begrijpen waarom die architectuur bij vertalen past.

De **encoder** verwerkt de volledige Engelse invoer en vormt contextafhankelijke representaties. Met self-attention kan een token informatie meenemen uit andere posities in dezelfde zin. De **decoder** gebruikt eerdere Nederlandse tokens om de volgende voorspelling te maken. Via cross-attention kan de decoder daarbij informatie uit de Engelse encoderrepresentatie gebruiken.

Omdat een attentionmechanisme op zichzelf geen vaste woordvolgorde oplegt, wordt ook een **positionele embedding** toegevoegd. In de decoder wordt causale self-attention gebruikt: bij een voorspelling mag het model niet naar toekomstige Nederlandse tokens kijken. Anders zou training informatie geven die tijdens het werkelijk vertalen nog niet beschikbaar is.

De architectuur heeft voordelen voor de opdracht: context tussen woorden kan direct worden gemodelleerd en sequenties kunnen tijdens training parallel worden verwerkt. Daar staan nadelen tegenover, zoals de rekenkosten van attention en de afhankelijkheid van voldoende en representatieve trainingsdata. Ook zegt het toepassen van attention op zichzelf niets over de feitelijke kwaliteit van vertalingen.

Ik neem de klassen `TransformerEncoder`, `TransformerDecoder` en `PositionalEmbedding` uit het lesvoorbeeld als uitgangspunt. Eerst laat ik het oorspronkelijke model werken; pas daarna onderzoek ik gericht mogelijke aanpassingen.

**Bronnen voor architectuur en modelwerking:**
- Vaswani et al. (2017), [*Attention Is All You Need*](https://arxiv.org/abs/1706.03762), introduceert de Transformerarchitectuur gebaseerd op attention.
- F. Chollet, [*Deep Learning with Python*](https://deeplearningwithpython.io/), hoofdstuk 15, zoals behandeld in les 3.
- [Keras MultiHeadAttention](https://keras.io/api/layers/attention_layers/multi_head_attention/), voor de werking van multi-head en causale attention.

### 5.1 Transformer Encoder

De encoder combineert self-attention met een feedforward-netwerk. Residual connections en layer normalization helpen de representaties door de opeenvolgende stappen te verwerken.

In [ ]:
class TransformerEncoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads, **kwargs):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, source, source_mask):
        residual = x = source
        mask = source_mask[:, None, :]
        x = self.self_attention(query=x, key=x, value=x, attention_mask=mask)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

### 5.2 Transformer Decoder

De decoder ontvangt de Nederlandse tokens die tot dat moment bekend zijn. Met *causal self-attention* kan hij geen toekomstige doelwoorden gebruiken. Met *cross-attention* kan hij tegelijk terugkijken naar de Engelse representatie uit de encoder. Dit is precies het onderscheid tussen een taalmodel dat alleen Nederlandse woorden voorspelt en een vertaalsysteem dat de Engelse invoer moet meenemen.

In [ ]:
class TransformerDecoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads, **kwargs):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.cross_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.cross_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, target, source, source_mask):
        residual = x = target
        x = self.self_attention(query=x, key=x, value=x, use_causal_mask=True)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        mask = source_mask[:, None, :]
        x = self.cross_attention(
            query=x, key=source, value=source, attention_mask=mask
        )
        x = x + residual
        x = self.cross_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

### 5.3 Positionele embeddings

De tokenembedding representeert welke woorden in een zin voorkomen; de positionele embedding geeft informatie over hun volgorde. Beide representaties worden in de voorbeeldimplementatie bij elkaar opgeteld. Die informatie is relevant, omdat het verwisselen van twee woorden de betekenis van een zin kan veranderen.

In [ ]:
class PositionalEmbedding(keras.Layer):
    def __init__(self, sequence_length, input_dim, output_dim, **kwargs):
        super().__init__()
        self.token_embeddings = layers.Embedding(input_dim, output_dim)
        self.position_embeddings = layers.Embedding(sequence_length, output_dim)

    def call(self, inputs):
        positions = ops.cumsum(ops.ones_like(inputs), axis=-1) - 1
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

### 5.4 De volledige architectuur samenstellen

In de code hieronder verbind ik de encoder en decoder met elkaar. De uitgangspunten uit les 3 zijn:

| Instelling | Baseline |
|---|---:|
| Vocabulaire per taal | 15.000 |
| Sequentielengte | 20 |
| Embedding-/hidden-dimensie | 256 |
| Feedforward-dimensie | 2056 |
| Attention-heads | 8 |
| Dropout vlak vóór de tokenvoorspelling | 0,5 |

De encoder krijgt een Engelse tokenreeks; de decoder krijgt de verschoven Nederlandse tokenreeks. De laatste Dense-laag produceert per decoderpositie een kansverdeling over de doelwoordenschat. Ik wil na het bouwen controleren hoeveel trainbare parameters het model bevat. Dat wordt later een nuttige maat voor de complexiteit wanneer ik de baseline met een verbetering vergelijk.

**Bewuste afbakening:** ik behoud de architectuur van de les, ook als een andere keuze theoretisch aantrekkelijk lijkt. Daardoor kan ik straks beter aangeven welk verschil voortkomt uit mijn eigen iteratie. De code gebruikt de naam `dutch` in plaats van de overgebleven naam `spanish` uit het lesvoorbeeld; dat verandert de werking niet.

In [ ]:
hidden_dim = 256
intermediate_dim = 2056
num_heads = 8

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

target = keras.Input(shape=(None,), dtype="int32", name="dutch")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions, name='Transformer_EN_NL_Baseline')
transformer.summary()
print('Aantal parameters:', transformer.count_params())

## 6. Het model compileren en trainen

Nu de data en het model zijn voorbereid, kan ik de baseline trainen. Ik gebruik **Adam** als optimizer en **sparse categorical crossentropy** als loss-functie, overeenkomstig les 3. Omdat de uitvoer uit token-ID's bestaat, is een categorische loss passend: voor iedere positie voorspelt het netwerk een kansverdeling over de Nederlandse woordenschat en wordt de kans van het juiste token beoordeeld.

Daarnaast registreer ik de **gewogen token-accuracy**. Die vertelt welk deel van de meegewogen doelposities correct wordt voorspeld. Dit is een bruikbare maat om te volgen of het netwerk leert, maar nog geen directe maat voor een goede volledige vertaling. Een zin kan grammaticaal of inhoudelijk onjuist zijn terwijl meerdere losse tokens wel correct worden voorspeld.

In de les is de `fit()`-aanroep uitgecommentarieerd en wordt zowel een verkorte test van twee epochs als een eerdere instelling van dertig epochs genoemd. Ik kies **30 epochs voor de definitieve baseline** om de ontwikkeling over meerdere trainingsrondes te kunnen bekijken. Een korte rooktest met twee epochs gebruik ik alleen om te controleren of de pipeline draait; die resultaten vergelijk ik niet met een volledig getraind verbeterd model.

Ik houd de testset buiten dit proces. De validatieloss en -accuracy laten zien hoe de modelprestaties zich ontwikkelen op niet-getrainde zinsparen. De uiteindelijke waarde van de baseline wordt pas na de uitvoering vastgesteld.

**Controlepunten:** seed, aantal epochs, optimizer, loss, batchgrootte, dataverdeling, trainingstijd en modelparameters. Een vaste seed beperkt variatie maar garandeert geen exact identieke uitkomst op andere systemen.

**Bron:** trainingsconfiguratie uit `Blok2_Les03.ipynb`; [Keras Model.fit](https://keras.io/api/models/model_training_apis/).

In [ ]:
EPOCHS = 30
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)

history = transformer.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    verbose=1,
)

transformer.save(MODEL_DIR / "transformer_baseline.keras")
with open(RESULTS_DIR / "transformer_baseline_history.json", "w", encoding="utf-8") as f:
    json.dump({key: [float(x) for x in values] for key, values in history.history.items()}, f, indent=2)
print("Model en trainingsgeschiedenis opgeslagen.")

## 7. Resultaten — training en validatie

Na de training wil ik niet alleen de eindscore bekijken, maar vooral **hoe die score tot stand komt**. Daarom zet ik voor iedere epoch de training- en validatieloss naast elkaar. Daarnaast bekijk ik de ontwikkeling van de token-accuracy.

Mijn eerste verwachting is dat de trainingsloss afneemt wanneer het model leert. Als de validatieloss op enig moment weer stijgt terwijl de trainingsloss blijft dalen, kan dat wijzen op overfitting. Wanneer beide losses hoog blijven, kan het model juist onvoldoende leren. Dat zijn mogelijke interpretaties, geen conclusies voordat ik de echte curves heb gezien.

Ik bewaar de grafieken en de onderliggende trainingsgeschiedenis, zodat ik bij deel 2 exact dezelfde baseline kan gebruiken in een vergelijking.

### Interpretatie van de grafieken — in te vullen na training

**Waarneming.** Wat is de laagste validatieloss en bij welke epoch wordt die bereikt? Hoe ontwikkelen train- en validatieloss zich ten opzichte van elkaar?

**Verklaring.** Welke verklaring past bij het waargenomen verloop, en welke alternatieve verklaringen blijven mogelijk?

**Gevolg voor de volgende iteratie.** Welke aanpassing zou dit specifieke patroon kunnen verbeteren? Ik kies geen verbetering voordat ik deze vragen met de gemeten resultaten kan beantwoorden.

In [ ]:
metrics = history.history
accuracy_key = next((k for k in metrics if k == "accuracy" or k.endswith("_accuracy") and not k.startswith("val_")), None)
val_accuracy_key = f"val_{accuracy_key}" if accuracy_key is not None else None
x = np.arange(1, len(metrics["loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(x, metrics["loss"], marker="o", markersize=2.5, label="Train loss")
axes[0].plot(x, metrics["val_loss"], marker="o", markersize=2.5, label="Validatie loss")
axes[0].set(xlabel="Epoch", ylabel="Sparse categorical crossentropy", title="Loss per epoch")
axes[0].legend()
axes[0].grid(alpha=0.2)

if accuracy_key and val_accuracy_key in metrics:
    axes[1].plot(x, metrics[accuracy_key], marker="o", markersize=2.5, label="Train token-accuracy")
    axes[1].plot(x, metrics[val_accuracy_key], marker="o", markersize=2.5, label="Validatie token-accuracy")
    axes[1].legend()
axes[1].set(xlabel="Epoch", ylabel="Gewogen token-accuracy", title="Accuracy per epoch")
axes[1].grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "03_learning_curves.png", dpi=160)
plt.show()

best_epoch = int(np.argmin(metrics["val_loss"])) + 1
print(f"Beste validatie-loss: {min(metrics['val_loss']):.4f} op epoch {best_epoch}")
if val_accuracy_key in metrics:
    print(f"Beste validatie-token-accuracy: {max(metrics[val_accuracy_key]):.4f}")
print("Let op: token-accuracy is geen maat voor volledige vertaalcorrectheid.")

## 8. Controleren of het model echt vertaalt

Een dalende loss betekent nog niet automatisch dat de Nederlandse zinnen inhoudelijk correct zijn. Daarom genereer ik na de training ook **voorbeeldvertalingen**. Ik vergelijk telkens de Engelse invoer, de Nederlandse referentiezin en de voorspelling van het model.

Deze controle is kwalitatief en beperkt. Er kunnen meerdere goede Nederlandse vertalingen bij dezelfde Engelse zin passen. Een letterlijke overeenkomst met de referentie is dus niet altijd nodig. Tegelijkertijd kunnen korte, veelvoorkomende woorden de token-accuracy verhogen terwijl de betekenis van de zin verkeerd wordt weergegeven.

Ik gebruik voor deze eerste inspectie een vaste kleine steekproef uit de **validatieset**. De testset blijft gereserveerd voor de eindbeoordeling. De decoder genereert tokens stap voor stap en kiest daarbij steeds het token met de hoogste voorspelde kans (*greedy decoding*).

Bij het bekijken van de uitkomsten let ik specifiek op:
- Of de kernbetekenis van de Engelse zin behouden blijft.
- Of woorden worden weggelaten, herhaald of verkeerd gekozen.
- Of langere zinnen meer fouten bevatten dan korte zinnen.
- Of de vertaling grammaticaal logisch is en op tijd stopt.

**Bron:** het vertaalvoorbeeld en de decoderlogica uit `Blok2_Les03.ipynb`.

In [ ]:
dutch_vocab = dutch_tokenizer.get_vocabulary()
dutch_index_lookup = dict(enumerate(dutch_vocab))

def translate_sentence(english_sentence):
    encoder_tokens = english_tokenizer([english_sentence])
    decoded = "[start]"
    for i in range(sequence_length):
        decoder_tokens = dutch_tokenizer([decoded])[:, :-1]
        probabilities = transformer.predict([encoder_tokens, decoder_tokens], verbose=0)
        token_id = int(np.argmax(probabilities[0, i, :]))
        token = dutch_index_lookup[token_id]
        if token in ("[end]", ""):
            break
        decoded += " " + token
    return decoded.replace("[start]", "", 1).strip()

sample_rng = random.Random(SEED)
examples = sample_rng.sample(val_pairs, k=min(8, len(val_pairs)))
rows = []
for english, reference in examples:
    rows.append({
        "Engels": english,
        "Referentie NL": reference.replace("[start]", "").replace("[end]", "").strip(),
        "Modelvertaling": translate_sentence(english),
    })
translations_df = pd.DataFrame(rows)
display(translations_df)
translations_df.to_csv(RESULTS_DIR / "voorbeeldvertalingen_validatie.csv", index=False)

print("Baseline gereed: controleer de learning curves en fouttypen vóór een modelverbetering.")

## 9. Tussenconclusie en verbeterpunten uit eerdere feedback

**Status: deze conclusies worden aangevuld nadat de training en de voorbeeldvertalingen zijn bekeken.** Op dit moment kan ik alleen vaststellen dat de gekozen methode een reproduceerbare basis voor vergelijking oplevert; of het model goed vertaalt, moet nog uit de metingen blijken.

Ik wil de feedback uit mijn eerdere M&T-opdrachten expliciet toepassen. Daarbij richt ik mij op:

| Leeruitkomst | Eerder verbeterpunt | Wat ik in deze opdracht anders doe |
|---|---|---|
| **A2** | Onvoldoende precies benoemd wanneer een model goed genoeg is. | Ik formuleer toetsbare eisen en koppel de uiteindelijke prestatie-eis aan de resultaten en de vertaaltaak. |
| **B1** | Validatiesplit onvoldoende verantwoord. | Ik motiveer de 70/15/15-verdeling, houd de testset apart en bespreek risico's van overlappende zinnen. |
| **B2** | Modelkeuzes soms onvoldoende afgeleid van taak en data. | Ik beschrijf waarom een encoder-decoder voor vertalen past en benoem ook de nadelen en beperkingen. |
| **B3** | Bij een iteratie veranderden meerdere factoren tegelijk. | Ik vergelijk later één gerichte aanpassing met dezelfde baseline en trainings-/evaluatieprocedure. |
| **C2** | Prestatiematen en fouten niet altijd diep genoeg beoordeeld. | Ik combineer learning curves, token-accuracy en een bespreking van concrete vertaalfouten. |

### Wat ik nog uit de resultaten moet halen

1. **Training:** [laagste validatieloss, epoch en eventuele overfitting invullen].
2. **Voorbeeldvertalingen:** [terugkerende correcte en onjuiste vertalingen beschrijven].
3. **Beperking:** [onderbouwen welke tekortkoming daadwerkelijk uit de resultaten blijkt].
4. **Vervolg:** [één gerichte hypothese formuleren voor deel 2].

Ik wil hiermee voorkomen dat de volgende iteratie alleen gebaseerd wordt op het idee dat een complexer model vanzelf beter is.

### Bronnen en herleidbaarheid

- **Lesbron:** `Blok2_Les03.ipynb`, *Sequence-to-sequence learning & the transformer*, code en lesuitleg.
- **Boekbron:** François Chollet, *Deep Learning with Python*, hoofdstuk 15, [deepLearningWithPython.io](https://deeplearningwithpython.io/). De opdracht verwijst naar de derde editie.
- **Architectuurbron:** Vaswani et al. (2017), [*Attention Is All You Need*](https://arxiv.org/abs/1706.03762).
- **Databron:** [ManyThings/Anki — Engels-Nederlandse zinsparen](https://www.manythings.org/anki/).
- **Technische documentatie:** [Keras TextVectorization](https://keras.io/api/layers/preprocessing_layers/text/text_vectorization/), [Keras MultiHeadAttention](https://keras.io/api/layers/attention_layers/multi_head_attention/) en [Keras Model.fit](https://keras.io/api/models/model_training_apis/).

Bij de definitieve versie controleer ik per bron welke specifieke bewering deze ondersteunt en voeg ik waar nodig de relevante passage toe. De eerdere feedback waarschuwde er immers voor om een bron niet alleen te noemen, maar ook inhoudelijk met de gemaakte keuze te verbinden.

#### Deel 1b
Bespreek de voor- en nadelen van de gebruikte dataset.

> Jouw antwoord hier

### Deel 2
We willen dit model nu verbeteren. Bedenk een (mogelijke) verbetering. Het is niet de bedoeling dat je bestaande pretrained modellen gebruikt. Train alles zelf.

#### Deel 2a
Stel een korte lijst van eisen op waar jouw verbeterde model aan hoort te voldoen. 

> Jouw antwoord hier

#### Deel 2b
Licht het volgende toe:
- Welke verbetering je uitvoert en waarom je denkt dat dit voor een verbetering zal zorgen.
- Welke maat je gebruikt om de verbetering te meten, en waarom.

Houd je antwoorden kort en bondig.

> Jouw antwoord hier

#### Deel 2c
Train dit verbeterde model.

### Deel 3
#### Deel 3a
Evalueer het model.

#### Deel 3b
Bespreek het volgende.
- Hoeveel je model verbeterd (of verslechterd) is
- In het geval van geen verbetering, een mogelijke verklaring waarom het niet werkte.
- Licht toe wat je volgende stap zou zijn om betere resultaten te halen. Deze stap hoef je niet uit te voeren, maar wees wel specifiek en verantwoord het met jouw resultaten.

Houd je antwoorden kort en bondig.

> Jouw antwoord hier